## License

Copyright HallResearch.ai 2026

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

**DISCLAIMER:** This notebook is not compliance or legal advice.

#### Imports



In [1]:
import os                       # to read environment variables
from getpass import getpass     # to enter API key securely if needed
from openai import OpenAI       # OpenAI API Python client
import pandas as pd             # basic data handling

#### Connect to OpenAI API

Set `OPENAI_API_KEY` as an environment variable before running the notebook, or paste it into the secure prompt when asked. The prompt does not display the key as you type.

In [2]:
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    api_key = getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=api_key)

Enter your OpenAI API key:  ········


#### Small Toy Benchmark
* Adapt to your domain
* Add enough questions for statistical significance
* Rerun multiple times to measure empirical performance variance



In [3]:
example_questions = [

    {
        "question": "What is the capital of France?",
        "choices": ["A. Berlin", "B. Madrid", "C. Paris", "D. Rome"],
        "answer": "C",
    },
    {
        "question": "2 + 2 * 3 = ?",
        "choices": ["A. 8", "B. 10", "C. 12", "D. 6"],
        "answer": "A",
    },
    {
        "question": "Which planet is known as the Red Planet?",
        "choices": ["A. Venus", "B. Mars", "C. Jupiter", "D. Saturn"],
        "answer": "B",
    },
    {
        "question": "Water freezes at what temperature at standard pressure?",
        "choices": ["A. 0 C", "B. 32 C", "C. 100 C", "D. -10 C"],
        "answer": "A",
    },
]


#### Utility Function to Submit Benchmark Questions to OpenAI API

In [4]:
def ask_model(question, choices, model="gpt-5-nano"):

  """
  Ask a model a multiple-choice question and return both a normalized prediction
  and the raw text response.

  Parameters
  ----------
  question : str
      The question to present to the model.
  choices : list[str]
      A list of answer choices, formatted
      ["A. ...", "B. ...", "C. ...", "D. ..."].
  model : str, optional
      The OpenAI model name to evaluate, by default "gpt-5-nano".

  Returns
  -------
  tuple(str, str)
      A 2-tuple of:
      - pred: the normalized predicted label, taken as the first character
        of the model output and uppercased
      - text: the raw text returned by the model
  """

  response = client.responses.create(
      model=model,
      instructions=( # system prompt/instructions
          "You are taking a multiple-choice benchmark. "
          "Return only one uppercase letter: A, B, C, or D."
      ),
      # benchmark question
      input=f"""
Question:
{question}

Choices:
{chr(10).join(choices)}
""".strip()
  )

  # simple output guardrail:
  # keep only the first character if the model returns extra text
  text = response.output_text.strip()
  pred = text[:1].upper()

  return pred, text



#### Run Benchmark and Assess Performance

In [5]:
# init scoring loop
rows = []
correct = 0

# run scoring loop
for ex in example_questions:

    pred, raw = ask_model(ex["question"], ex["choices"]) # ask questions with choices
    is_correct = pred == ex["answer"] # test answer
    correct += int(is_correct) # tally number of correct answers

    # add row to output dataset
    rows.append({
        "question": ex["question"],
        "prediction": pred,
        "gold": ex["answer"],
        "correct": is_correct,
        "raw_output": raw,
    })

results = pd.DataFrame(rows) # finalize output dataset
accuracy = correct / len(example_questions) # calculate accuracy

# basic output
print(results)
print(f"\nAccuracy: {accuracy:.2%}")

                                            question prediction gold  correct  \
0                     What is the capital of France?          C    C     True   
1                                      2 + 2 * 3 = ?          A    A     True   
2           Which planet is known as the Red Planet?          B    B     True   
3  Water freezes at what temperature at standard ...          A    A     True   

  raw_output  
0          C  
1          A  
2          B  
3          A  

Accuracy: 100.00%
